# Interactive FITS File Viewer

This notebook allows you to browse through FITS files in a folder by providing a path and using navigation buttons.

## Import Required Libraries

In [1]:
import os
from pathlib import Path
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
import ipywidgets as widgets
from IPython.display import display, clear_output

## Define Source Discovery and Data Loading Functions

In [2]:
RADCAT_F_PATH = "/home/abigaildeklerk/Desktop/DeKlerk_Models/Data/RADCAT_F"
SURVEYS = ['FIRST', 'LoTSS', 'NVSS']

def get_source_ids():
    """
    Discover all available source IDs from RADCAT_F.
    
    Returns:
        list: Sorted list of source IDs
    """
    source_ids = set()
    
    for survey in SURVEYS:
        survey_path = os.path.join(RADCAT_F_PATH, survey)
        if os.path.exists(survey_path):
            for file in os.listdir(survey_path):
                if file.lower().endswith(('.fits', '.fit')):
                    source_id = os.path.splitext(file)[0]
                    source_ids.add(source_id)
    
    return sorted(list(source_ids))

def get_fits_file(survey, source_id):
    """
    Get the FITS file path for a given survey and source ID.
    
    Args:
        survey (str): Survey name (FIRST, LoTSS, or NVSS)
        source_id (str): Source ID
        
    Returns:
        str: Path to FITS file or None if not found
    """
    file_path = os.path.join(RADCAT_F_PATH, survey, f"{source_id}.fits")
    if os.path.exists(file_path):
        return file_path
    return None

## Handle FITS File Loading and Display

In [3]:
def display_source_surveys(source_id):
    """
    Load and display FITS files from all three surveys for a given source.
    
    Args:
        source_id (str): Source ID
    """
    try:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Source {source_id}: All Surveys', fontsize=14, fontweight='bold')
        
        survey_colors = {'FIRST': 'hot', 'LoTSS': 'viridis', 'NVSS': 'plasma'}
        
        for idx, survey in enumerate(SURVEYS):
            ax = axes[idx]
            fits_path = get_fits_file(survey, source_id)
            
            if fits_path is None:
                ax.text(0.5, 0.5, f'{survey} file not found', 
                       ha='center', va='center', transform=ax.transAxes, fontsize=12)
                ax.set_title(f'{survey}', fontsize=12)
                ax.axis('off')
                continue
            
            try:
                with fits.open(fits_path) as hdul:
                    data = hdul[0].data
                    
                    if data is None:
                        for hdu in hdul[1:]:
                            if hdu.data is not None:
                                data = hdu.data
                                break
                    
                    if data is None:
                        ax.text(0.5, 0.5, f'No image data', 
                               ha='center', va='center', transform=ax.transAxes, fontsize=12)
                        ax.set_title(f'{survey}', fontsize=12)
                        ax.axis('off')
                        continue
                    
                    # Handle 2D or 3D data
                    if data.ndim == 3:
                        image_data = data[0]
                    else:
                        image_data = data
                    
                    # Use log scale for better visualization
                    image_data = np.nan_to_num(image_data, nan=0, posinf=0, neginf=0)
                    
                    # Determine normalization
                    if np.all(image_data >= 0):
                        norm = LogNorm(vmin=np.nanmax(image_data) / 1e6, vmax=np.nanmax(image_data))
                    else:
                        norm = SymLogNorm(linthresh=0.03, vmin=np.nanmin(image_data), vmax=np.nanmax(image_data))
                    
                    im = ax.imshow(image_data, cmap=survey_colors[survey], norm=norm, origin='lower')
                    ax.set_title(f'{survey}', fontsize=12, fontweight='bold')
                    ax.set_xlabel('X')
                    ax.set_ylabel('Y')
                    plt.colorbar(im, ax=ax, label='Intensity')
                    
            except Exception as e:
                ax.text(0.5, 0.5, f'Error loading:\n{str(e)}', 
                       ha='center', va='center', transform=ax.transAxes, fontsize=10)
                ax.set_title(f'{survey}', fontsize=12)
                ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Error displaying source {source_id}: {str(e)}")

## Create Interactive Display Widget with Navigation Controls

In [ ]:
# Get available source IDs
source_ids = get_source_ids()

if not source_ids:
    print("No sources found in RADCAT_F!")
else:
    # State variables
    current_index = 0
    
    # Create widgets
    source_dropdown = widgets.Dropdown(
        options=source_ids,
        value=source_ids[0],
        description='Source ID:',
        style={'description_width': '80px'}
    )
    
    prev_button = widgets.Button(
        description='← Previous',
        button_style='info',
        tooltip='Show previous source'
    )
    
    next_button = widgets.Button(
        description='Next →',
        button_style='info',
        tooltip='Show next source'
    )
    
    status_label = widgets.HTML(value=f'<b>Status:</b> Showing {source_ids[0]} (1 of {len(source_ids)})')
    
    display_output = widgets.Output()
    
    def on_prev_clicked(b):
        """Handle previous button click."""
        global current_index
        current_index = (current_index - 1) % len(source_ids)
        source_dropdown.value = source_ids[current_index]
    
    def on_next_clicked(b):
        """Handle next button click."""
        global current_index
        current_index = (current_index + 1) % len(source_ids)
        source_dropdown.value = source_ids[current_index]
    
    def on_dropdown_changed(change):
        """Handle source dropdown change."""
        global current_index
        current_index = source_ids.index(change['new'])
        update_display()
    
    def update_display():
        """Update the display with the selected source."""
        source_id = source_dropdown.value
        current_num = current_index + 1
        status_label.value = f'<b>Status:</b> Showing {source_id} ({current_num} of {len(source_ids)})'
        
        with display_output:
            clear_output(wait=True)
            display_source_surveys(source_id)
    
    # Attach event handlers
    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)
    source_dropdown.observe(on_dropdown_changed, names='value')
    
    # Create layout
    dropdown_box = widgets.HBox([source_dropdown], layout=widgets.Layout(justify_content='center', margin='10px'))
    nav_buttons = widgets.HBox([prev_button, next_button], layout=widgets.Layout(justify_content='center', margin='10px'))
    
    # Display the interface
    print("=" * 80)
    print("FITS FILE VIEWER - All Surveys Side by Side")
    print("=" * 80)
    display(dropdown_box)
    display(nav_buttons)
    display(status_label)
    display(display_output)
    
    # Show initial display
    update_display()

FITS FILE VIEWER - All Surveys Side by Side


HTML(value='<b>Status:</b> Showing 10000 (1 of 2510)')

Output()